In [1]:
# ============================================================
# POSTGRESQL CONNECTION SETUP
# ============================================================

# Install required packages
%pip install -q sqlalchemy psycopg2-binary python-dotenv

import os
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


# ------------------------------------------------------------
# 1. Define project root
# ------------------------------------------------------------

PROJECT_ROOT = Path("/home/jovyan/work")


# ------------------------------------------------------------
# 2. Create local .env file if it does not already exist
# ------------------------------------------------------------

env_path = PROJECT_ROOT / ".env"

if not env_path.exists():

    env_path.write_text(
        "POSTGRES_USER=nhs_user\n"
        "POSTGRES_PASSWORD=nhs_password\n"
        "POSTGRES_DB=nhs_analytics\n"
        "POSTGRES_HOST=host.docker.internal\n"
        "POSTGRES_PORT=5432\n"
    )

    print(".env file created.")

else:
    print(".env file already exists.")


# ------------------------------------------------------------
# 3. Load database credentials
# ------------------------------------------------------------

load_dotenv(env_path)

POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_PORT = int(os.getenv("POSTGRES_PORT"))


# ------------------------------------------------------------
# 4. Create PostgreSQL connection URL
# ------------------------------------------------------------

database_url = URL.create(
    drivername="postgresql+psycopg2",
    username=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    database=POSTGRES_DB
)


# ------------------------------------------------------------
# 5. Create SQLAlchemy database engine
# ------------------------------------------------------------

engine = create_engine(database_url)


# ------------------------------------------------------------
# 6. Test the PostgreSQL connection
# ------------------------------------------------------------

with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT
                current_database() AS database_name,
                current_user AS database_user;
        """)
    )

    row = result.fetchone()


print("=" * 50)
print("POSTGRESQL CONNECTION TEST")
print("=" * 50)

print("Database:", row.database_name)
print("User:", row.database_user)

print("\nConnection successful.")

Note: you may need to restart the kernel to use updated packages.
.env file created.
POSTGRESQL CONNECTION TEST
Database: nhs_analytics
User: nhs_user

Connection successful.


In [3]:
# ============================================================
# FIX SQL COLUMN NAMES AND RELOAD POSTGRESQL TABLES
# ============================================================

import re
import pandas as pd
from sqlalchemy import text


# ------------------------------------------------------------
# 1. Define processed data path
# ------------------------------------------------------------

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"


# ------------------------------------------------------------
# 2. Map CSV files to PostgreSQL tables
# ------------------------------------------------------------

table_files = {
    "ae_monthly_summary": "ae_monthly_summary.csv",
    "ae_provider_month": "ae_provider_month.csv",
    "ae_provider_summary": "ae_provider_summary.csv",
    "ae_type1_provider_month": "ae_type1_provider_month.csv",
    "ae_type1_provider_summary": "ae_type1_provider_summary.csv"
}


# ------------------------------------------------------------
# 3. Create better SQL-friendly column names
# ------------------------------------------------------------

def clean_column_name(column_name):

    # Convert to lowercase and remove extra spaces
    column_name = column_name.strip().lower()

    # Standardise common NHS terms
    column_name = column_name.replace("a&e", "ae")
    column_name = column_name.replace("4 hour", "four_hour")

    # Replace symbols with readable text
    column_name = column_name.replace("%", "pct")
    column_name = column_name.replace("&", "and")

    # Replace remaining special characters and spaces
    column_name = re.sub(
        r"[^a-z0-9]+",
        "_",
        column_name
    )

    # Remove leading and trailing underscores
    column_name = column_name.strip("_")

    return column_name


# ------------------------------------------------------------
# 4. Reload all processed CSV files into PostgreSQL
# ------------------------------------------------------------

loaded_tables = {}

print("=" * 70)
print("RELOADING CLEAN A&E TABLES INTO POSTGRESQL")
print("=" * 70)

for table_name, file_name in table_files.items():

    file_path = PROCESSED_PATH / file_name

    # Load CSV
    df = pd.read_csv(file_path)

    # Clean column names
    df.columns = [
        clean_column_name(column)
        for column in df.columns
    ]

    # Convert month to datetime where available
    if "month" in df.columns:
        df["month"] = pd.to_datetime(df["month"])

    # Check for duplicate column names after cleaning
    if df.columns.duplicated().any():

        duplicate_columns = df.columns[
            df.columns.duplicated()
        ].tolist()

        raise ValueError(
            f"Duplicate columns found in {table_name}: "
            f"{duplicate_columns}"
        )

    # Replace existing PostgreSQL table
    df.to_sql(
        name=table_name,
        con=engine,
        schema="public",
        if_exists="replace",
        index=False,
        chunksize=1000,
        method="multi"
    )

    loaded_tables[table_name] = len(df)

    print(
        f"{table_name:<30} "
        f"Rows: {len(df):>6} | "
        f"Columns: {len(df.columns):>2}"
    )


# ------------------------------------------------------------
# 5. Show columns in ae_monthly_summary
# ------------------------------------------------------------

column_query = text("""
    SELECT
        ordinal_position,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = 'ae_monthly_summary'
    ORDER BY ordinal_position;
""")


with engine.connect() as connection:

    monthly_columns = pd.read_sql(
        column_query,
        connection
    )


print("\n" + "=" * 70)
print("AE_MONTHLY_SUMMARY COLUMN NAMES")
print("=" * 70)

display(monthly_columns)


# ------------------------------------------------------------
# 6. Validate Python rows against PostgreSQL rows
# ------------------------------------------------------------

validation_results = []

with engine.connect() as connection:

    for table_name, python_rows in loaded_tables.items():

        postgres_rows = connection.execute(
            text(
                f'SELECT COUNT(*) '
                f'FROM public."{table_name}"'
            )
        ).scalar()

        validation_results.append({
            "table_name": table_name,
            "python_rows": python_rows,
            "postgres_rows": postgres_rows,
            "status": (
                "PASS"
                if python_rows == postgres_rows
                else "FAIL"
            )
        })


validation_df = pd.DataFrame(validation_results)


print("\n" + "=" * 70)
print("ROW COUNT VALIDATION")
print("=" * 70)

display(validation_df)


# ------------------------------------------------------------
# 7. Run first real SQL query
# ------------------------------------------------------------

test_query = text("""
    SELECT
        month,
        attendances,
        four_hour_performance_pct,
        admission_rate_pct
    FROM ae_monthly_summary
    ORDER BY month
    LIMIT 5;
""")


with engine.connect() as connection:

    test_result = pd.read_sql(
        test_query,
        connection
    )


print("\n" + "=" * 70)
print("FIRST SUCCESSFUL SQL QUERY")
print("=" * 70)

display(test_result)


# ------------------------------------------------------------
# 8. Final check
# ------------------------------------------------------------

if (validation_df["status"] == "PASS").all():

    print("\nAll PostgreSQL tables passed row-count validation.")

else:

    print("\nWarning: Some tables failed validation.")

RELOADING CLEAN A&E TABLES INTO POSTGRESQL
ae_monthly_summary             Rows:     29 | Columns: 13
ae_provider_month              Rows:   5714 | Columns:  9
ae_provider_summary            Rows:    219 | Columns:  9
ae_type1_provider_month        Rows:   5714 | Columns:  6
ae_type1_provider_summary      Rows:    219 | Columns:  7

AE_MONTHLY_SUMMARY COLUMN NAMES


,ordinal_position,column_name,data_type
0,1,attendances,bigint
1,2,over_four_hours,bigint
2,3,four_hour_performance_pct,double precision
3,4,admissions_via_ae,bigint
4,5,file_name,text
5,6,month_text,text
6,7,month,timestamp without time zone
7,8,days_in_month,bigint
8,9,attendances_per_day,double precision
9,10,admission_rate_pct,double precision



ROW COUNT VALIDATION


,table_name,python_rows,postgres_rows,status
0,ae_monthly_summary,29,29,PASS
1,ae_provider_month,5714,5714,PASS
2,ae_provider_summary,219,219,PASS
3,ae_type1_provider_month,5714,5714,PASS
4,ae_type1_provider_summary,219,219,PASS



FIRST SUCCESSFUL SQL QUERY


,month,attendances,four_hour_performance_pct,admission_rate_pct
0,2024-04-01,2163149,73.97,18.67
1,2024-05-01,2331010,73.43,18.03
2,2024-06-01,2215640,74.07,18.15
3,2024-07-01,2251330,74.76,18.29
4,2024-08-01,2086225,75.82,19.01



All PostgreSQL tables passed row-count validation.


In [4]:
# ============================================================
# SQL ANALYSIS - DATA VALIDATION AND BUSINESS QUERIES
# ============================================================

from sqlalchemy import text
import pandas as pd


# ------------------------------------------------------------
# Helper function to run SQL and display results
# ------------------------------------------------------------

def run_sql(query):
    with engine.connect() as connection:
        return pd.read_sql(text(query), connection)


# ============================================================
# 1. CHECK PROVIDER NAME CONSISTENCY
# ============================================================

provider_name_check = run_sql("""
    SELECT
        org_code,
        COUNT(DISTINCT org_name) AS different_names,
        MIN(org_name) AS example_name_1,
        MAX(org_name) AS example_name_2
    FROM ae_provider_month
    GROUP BY org_code
    HAVING COUNT(DISTINCT org_name) > 1
    ORDER BY different_names DESC, org_code;
""")


print("=" * 70)
print("PROVIDER NAME CONSISTENCY CHECK")
print("=" * 70)

print(
    "Organisation codes with multiple names:",
    len(provider_name_check)
)

display(provider_name_check)


# ============================================================
# 2. NATIONAL MONTHLY SUMMARY
# ============================================================

monthly_summary_sql = run_sql("""
    SELECT
        month,
        attendances,
        over_four_hours,
        four_hour_performance_pct,
        admissions_via_ae,
        admission_rate_pct
    FROM ae_monthly_summary
    ORDER BY month;
""")


print("\n" + "=" * 70)
print("NATIONAL MONTHLY A&E SUMMARY")
print("=" * 70)

display(monthly_summary_sql.head(10))


# ============================================================
# 3. TOP 5 HIGHEST-DEMAND MONTHS
# ============================================================

highest_demand_months = run_sql("""
    SELECT
        month,
        attendances,
        attendances_per_day,
        four_hour_performance_pct
    FROM ae_monthly_summary
    ORDER BY attendances DESC
    LIMIT 5;
""")


print("\n" + "=" * 70)
print("TOP 5 HIGHEST-DEMAND MONTHS")
print("=" * 70)

display(highest_demand_months)


# ============================================================
# 4. TOP 5 LOWEST-DEMAND MONTHS
# ============================================================

lowest_demand_months = run_sql("""
    SELECT
        month,
        attendances,
        attendances_per_day,
        four_hour_performance_pct
    FROM ae_monthly_summary
    ORDER BY attendances ASC
    LIMIT 5;
""")


print("\n" + "=" * 70)
print("TOP 5 LOWEST-DEMAND MONTHS")
print("=" * 70)

display(lowest_demand_months)


# ============================================================
# 5. BEST 4-HOUR PERFORMANCE MONTHS
# ============================================================

best_performance_months = run_sql("""
    SELECT
        month,
        attendances,
        four_hour_performance_pct,
        over_four_hours
    FROM ae_monthly_summary
    ORDER BY four_hour_performance_pct DESC
    LIMIT 5;
""")


print("\n" + "=" * 70)
print("TOP 5 BEST 4-HOUR PERFORMANCE MONTHS")
print("=" * 70)

display(best_performance_months)


# ============================================================
# 6. WORST 4-HOUR PERFORMANCE MONTHS
# ============================================================

worst_performance_months = run_sql("""
    SELECT
        month,
        attendances,
        four_hour_performance_pct,
        over_four_hours
    FROM ae_monthly_summary
    ORDER BY four_hour_performance_pct ASC
    LIMIT 5;
""")


print("\n" + "=" * 70)
print("TOP 5 WORST 4-HOUR PERFORMANCE MONTHS")
print("=" * 70)

display(worst_performance_months)


# ============================================================
# 7. YEARLY PERFORMANCE SUMMARY
# ============================================================

yearly_summary = run_sql("""
    SELECT
        EXTRACT(YEAR FROM month)::INTEGER AS year,

        SUM(attendances) AS total_attendances,

        ROUND(
            AVG(attendances_per_day)::NUMERIC,
            0
        ) AS average_daily_attendances,

        ROUND(
            AVG(four_hour_performance_pct)::NUMERIC,
            2
        ) AS average_four_hour_performance_pct,

        SUM(admissions_via_ae) AS total_admissions

    FROM ae_monthly_summary

    GROUP BY
        EXTRACT(YEAR FROM month)

    ORDER BY year;
""")


print("\n" + "=" * 70)
print("YEARLY A&E SUMMARY")
print("=" * 70)

display(yearly_summary)


# ============================================================
# 8. MONTHS WITH BELOW-72% PERFORMANCE
# ============================================================

poor_performance_months = run_sql("""
    SELECT
        month,
        attendances,
        four_hour_performance_pct,
        over_four_hours
    FROM ae_monthly_summary
    WHERE four_hour_performance_pct < 72
    ORDER BY four_hour_performance_pct ASC;
""")


print("\n" + "=" * 70)
print("MONTHS WITH PERFORMANCE BELOW 72%")
print("=" * 70)

display(poor_performance_months)

PROVIDER NAME CONSISTENCY CHECK
Organisation codes with multiple names: 12


,org_code,different_names,example_name_1,example_name_2
0,A5D4Q,2,NEMS CBS (UTC),NEMS UTC
1,NNJ15,2,OAKHAM,RUTLAND MINOR ILLNESS AND INJURY SERVICE (FORM...
2,RA2,2,ROYAL SURREY COUNTY HOSPITAL NHS FOUNDATION TRUST,ROYAL SURREY NHS FOUNDATION TRUST
3,RA7,2,BRISTOL NHS FOUNDATION TRUST,UNIVERSITY HOSPITALS BRISTOL AND WESTON NHS FO...
4,RAX,2,KINGSTON AND RICHMOND NHS FOUNDATION TRUST,KINGSTON HOSPITAL NHS FOUNDATION TRUST
5,RHU,2,PORTSMOUTH HOSPITALS UNIVERSITY NATIONAL HEALT...,PORTSMOUTH HOSPITALS UNIVERSITY NHS TRUST
6,RRF,2,"WRIGHTINGTON, WIGAN AND LEIGH NHS FOUNDATION T...","WRIGHTINGTON, WIGAN AND LEIGH TEACHING HOSPITA..."
7,RW1,2,HAMPSHIRE AND ISLE OF WIGHT HEALTHCARE NHS FOU...,SOUTHERN HEALTH NHS FOUNDATION TRUST
8,RWD,2,UNITED LINCOLNSHIRE HOSPITALS NHS TRUST,UNITED LINCOLNSHIRE TEACHING HOSPITALS NHS TRUST
9,RWH,2,EAST AND NORTH HERTFORDSHIRE NHS TRUST,EAST AND NORTH HERTFORDSHIRE TEACHING NHS TRUST



NATIONAL MONTHLY A&E SUMMARY


,month,attendances,over_four_hours,four_hour_performance_pct,admissions_via_ae,admission_rate_pct
0,2024-04-01,2163149,563068,73.97,403934,18.67
1,2024-05-01,2331010,619285,73.43,420204,18.03
2,2024-06-01,2215640,574514,74.07,402055,18.15
3,2024-07-01,2251330,568225,74.76,411782,18.29
4,2024-08-01,2086225,504447,75.82,396515,19.01
5,2024-09-01,2140977,563079,73.70,399406,18.66
6,2024-10-01,2280392,626992,72.51,422048,18.51
7,2024-11-01,2234578,634410,71.61,414109,18.53
8,2024-12-01,2258452,667235,70.46,417067,18.47
9,2025-01-01,2133300,589826,72.35,406228,19.04



TOP 5 HIGHEST-DEMAND MONTHS


...

,month,attendances,four_hour_performance_pct,over_four_hours
0,2024-12-01,2258452,70.46,667235
1,2024-11-01,2234578,71.61,634410
2,2026-01-01,2235696,71.83,629727
3,2025-01-01,2133300,72.35,589826
4,2024-10-01,2280392,72.51,626992



YEARLY A&E SUMMARY


,year,total_attendances,average_daily_attendances,average_four_hour_performance_pct,total_admissions
0,2024,19961753.0,72593.0,73.37,3687120.0
1,2025,26780401.0,73372.0,74.15,4829041.0
2,2026,18319952.0,75376.0,74.71,3167368.0



MONTHS WITH PERFORMANCE BELOW 72%


,month,attendances,four_hour_performance_pct,over_four_hours
0,2024-12-01,2258452,70.46,667235
1,2024-11-01,2234578,71.61,634410
2,2026-01-01,2235696,71.83,629727


In [5]:
# ============================================================
# ADVANCED SQL - PROVIDER ANALYSIS
# CTEs, RANK, LAG AND PROVIDER NAME STANDARDISATION
# ============================================================


# ------------------------------------------------------------
# 1. CREATE A CANONICAL PROVIDER VIEW
# ------------------------------------------------------------
# We use org_code as the stable identifier.
# The latest observed organisation name is used for display.

with engine.begin() as connection:

    connection.execute(text("""
        CREATE OR REPLACE VIEW provider_dimension AS

        SELECT DISTINCT ON (org_code)

            org_code,
            org_name AS latest_org_name,
            month AS latest_reporting_month

        FROM ae_provider_month

        WHERE org_code IS NOT NULL

        ORDER BY
            org_code,
            month DESC;
    """))


print("provider_dimension view created successfully.")


# ------------------------------------------------------------
# 2. CHECK THE CANONICAL PROVIDER VIEW
# ------------------------------------------------------------

provider_dimension = run_sql("""
    SELECT
        org_code,
        latest_org_name,
        latest_reporting_month
    FROM provider_dimension
    ORDER BY org_code
    LIMIT 10;
""")


print("\n" + "=" * 70)
print("CANONICAL PROVIDER DIMENSION")
print("=" * 70)

display(provider_dimension)


# ============================================================
# 3. CREATE PROVIDER PERFORMANCE SUMMARY USING A CTE
# ============================================================

provider_performance = run_sql("""
    WITH provider_totals AS (

        SELECT
            org_code,

            COUNT(DISTINCT month) AS months_reported,

            SUM(total_ae_attendances) AS total_attendances,

            SUM(total_over_four_hours) AS total_over_four_hours,

            SUM(admissions_via_ae) AS total_admissions

        FROM ae_provider_month

        WHERE total_ae_attendances > 0

        GROUP BY org_code
    )

    SELECT
        p.org_code,

        d.latest_org_name AS org_name,

        p.months_reported,

        p.total_attendances,

        ROUND(
            (
                (
                    p.total_attendances
                    - p.total_over_four_hours
                )::NUMERIC
                / p.total_attendances
            ) * 100,
            2
        ) AS four_hour_performance_pct,

        ROUND(
            (
                p.total_admissions::NUMERIC
                / p.total_attendances
            ) * 100,
            2
        ) AS admission_rate_pct

    FROM provider_totals p

    LEFT JOIN provider_dimension d
        ON p.org_code = d.org_code

    ORDER BY
        p.total_attendances DESC;
""")


print("\n" + "=" * 70)
print("PROVIDER PERFORMANCE SUMMARY")
print("=" * 70)

display(provider_performance.head(10))


# ============================================================
# 4. RANK THE BUSIEST PROVIDERS
# ============================================================

provider_ranking = run_sql("""
    WITH provider_totals AS (

        SELECT
            org_code,
            SUM(total_ae_attendances) AS total_attendances

        FROM ae_provider_month

        WHERE total_ae_attendances > 0

        GROUP BY org_code
    ),

    ranked_providers AS (

        SELECT
            org_code,
            total_attendances,

            RANK() OVER (
                ORDER BY total_attendances DESC
            ) AS attendance_rank

        FROM provider_totals
    )

    SELECT
        r.attendance_rank,
        r.org_code,
        d.latest_org_name AS org_name,
        r.total_attendances

    FROM ranked_providers r

    LEFT JOIN provider_dimension d
        ON r.org_code = d.org_code

    WHERE r.attendance_rank <= 10

    ORDER BY r.attendance_rank;
""")


print("\n" + "=" * 70)
print("TOP 10 BUSIEST PROVIDERS - SQL RANK")
print("=" * 70)

display(provider_ranking)


# ============================================================
# 5. IDENTIFY HIGH-VOLUME WEAK PERFORMERS
# ============================================================

high_volume_weak_performers = run_sql("""
    WITH provider_totals AS (

        SELECT
            org_code,

            COUNT(DISTINCT month) AS months_reported,

            SUM(total_ae_attendances) AS total_attendances,

            SUM(total_over_four_hours) AS total_over_four_hours

        FROM ae_provider_month

        WHERE total_ae_attendances > 0

        GROUP BY org_code
    ),

    provider_metrics AS (

        SELECT
            org_code,
            months_reported,
            total_attendances,

            total_attendances::NUMERIC
            / months_reported
                AS average_monthly_attendances,

            (
                (
                    total_attendances
                    - total_over_four_hours
                )::NUMERIC
                / total_attendances
            ) * 100
                AS four_hour_performance_pct

        FROM provider_totals

        WHERE months_reported >= 24
    ),

    demand_threshold AS (

        SELECT
            PERCENTILE_CONT(0.75)
            WITHIN GROUP (
                ORDER BY average_monthly_attendances
            ) AS high_volume_cutoff

        FROM provider_metrics
    )

    SELECT
        p.org_code,

        d.latest_org_name AS org_name,

        ROUND(
            p.average_monthly_attendances,
            0
        ) AS average_monthly_attendances,

        ROUND(
            p.four_hour_performance_pct,
            2
        ) AS four_hour_performance_pct

    FROM provider_metrics p

    CROSS JOIN demand_threshold t

    LEFT JOIN provider_dimension d
        ON p.org_code = d.org_code

    WHERE
        p.average_monthly_attendances
        >= t.high_volume_cutoff

    ORDER BY
        p.four_hour_performance_pct ASC

    LIMIT 10;
""")


print("\n" + "=" * 70)
print("HIGH-VOLUME PROVIDERS WITH WEAKEST PERFORMANCE")
print("=" * 70)

display(high_volume_weak_performers)


# ============================================================
# 6. PROVIDER MONTH-ON-MONTH ANALYSIS USING LAG()
# ============================================================

provider_mom = run_sql("""
    WITH monthly_provider_activity AS (

        SELECT
            month,
            org_code,
            total_ae_attendances,
            four_hour_performance_pct

        FROM ae_provider_month

        WHERE total_ae_attendances > 0
    ),

    previous_month AS (

        SELECT
            month,
            org_code,
            total_ae_attendances,
            four_hour_performance_pct,

            LAG(total_ae_attendances) OVER (
                PARTITION BY org_code
                ORDER BY month
            ) AS previous_month_attendances,

            LAG(four_hour_performance_pct) OVER (
                PARTITION BY org_code
                ORDER BY month
            ) AS previous_month_performance

        FROM monthly_provider_activity
    )

    SELECT
        p.month,

        p.org_code,

        d.latest_org_name AS org_name,

        p.total_ae_attendances,

        p.previous_month_attendances,

        ROUND(
            (
                (
                    p.total_ae_attendances
                    - p.previous_month_attendances
                )::NUMERIC
                / NULLIF(
                    p.previous_month_attendances,
                    0
                )
            ) * 100,
            2
        ) AS attendance_mom_change_pct,

        p.four_hour_performance_pct,

        ROUND(
            (
                p.four_hour_performance_pct
                - p.previous_month_performance
            )::NUMERIC,
            2
        ) AS performance_mom_change_pp

    FROM previous_month p

    LEFT JOIN provider_dimension d
        ON p.org_code = d.org_code

    WHERE p.previous_month_attendances IS NOT NULL

    ORDER BY
        ABS(
            (
                (
                    p.total_ae_attendances
                    - p.previous_month_attendances
                )::NUMERIC
                / NULLIF(
                    p.previous_month_attendances,
                    0
                )
            ) * 100
        ) DESC

    LIMIT 20;
""")


print("\n" + "=" * 70)
print("LARGEST PROVIDER MONTH-ON-MONTH CHANGES")
print("=" * 70)

display(provider_mom)


# ============================================================
# 7. SAVE SQL RESULTS FOR LATER USE
# ============================================================

provider_performance.to_csv(
    PROCESSED_PATH / "sql_provider_performance.csv",
    index=False
)

provider_ranking.to_csv(
    PROCESSED_PATH / "sql_top_10_provider_ranking.csv",
    index=False
)

high_volume_weak_performers.to_csv(
    PROCESSED_PATH / "sql_high_volume_weak_performers.csv",
    index=False
)

print("\nSQL provider outputs saved successfully.")

provider_dimension view created successfully.

CANONICAL PROVIDER DIMENSION


,org_code,latest_org_name,latest_reporting_month
0,8J094,BADGER LTD,2026-03-01
1,A5D4Q,NEMS CBS (UTC),2025-09-01
2,AAH,TETBURY HOSPITAL TRUST LTD,2026-08-01
3,ACH01,WHITSTABLE MEDICAL PRACTICE,2026-08-01
4,AD913,BECKENHAM BEACON UCC,2026-08-01
5,AJN,WORKINGTON HEALTH LIMITED,2026-08-01
6,AQM01,FAVERSHAM MEDICAL PRACTICE MIU,2026-08-01
7,AQN04,PHL LYMINGTON UTC,2026-08-01
8,ARN03,LOCAL CARE DIRECT,2026-08-01
9,AXG,WILTSHIRE HEALTH & CARE,2025-02-01



PROVIDER PERFORMANCE SUMMARY


,org_code,org_name,months_reported,total_attendances,four_hour_performance_pct,admission_rate_pct
0,R1H,BARTS HEALTH NHS TRUST,29,1280346.0,71.03,10.01
1,R0A,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,29,1155337.0,71.66,15.07
2,RRK,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,29,1115793.0,63.45,27.63
3,RAL,ROYAL FREE LONDON NHS FOUNDATION TRUST,29,1082166.0,76.95,14.53
4,RDU,FRIMLEY HEALTH NHS FOUNDATION TRUST,29,1019373.0,72.79,17.91
5,RAJ,MID AND SOUTH ESSEX NHS FOUNDATION TRUST,29,1016100.0,71.10,25.36
6,RM3,NORTHERN CARE ALLIANCE NHS FOUNDATION TRUST,29,1014573.0,68.93,21.99
7,RF4,"BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...",29,867005.0,78.49,21.68
8,RR8,LEEDS TEACHING HOSPITALS NHS TRUST,29,865253.0,75.79,12.32
9,RYR,UNIVERSITY HOSPITALS SUSSEX NHS FOUNDATION TRUST,29,863174.0,64.22,28.61



TOP 10 BUSIEST PROVIDERS - SQL RANK


...


HIGH-VOLUME PROVIDERS WITH WEAKEST PERFORMANCE


,org_code,org_name,average_monthly_attendances,four_hour_performance_pct
0,RX1,NOTTINGHAM UNIVERSITY HOSPITALS NHS TRUST,19607.0,59.28
1,RWE,UNIVERSITY HOSPITALS OF LEICESTER NHS TRUST,24080.0,61.99
2,RRK,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,38476.0,63.45
3,RYR,UNIVERSITY HOSPITALS SUSSEX NHS FOUNDATION TRUST,29765.0,64.22
4,RWP,WORCESTERSHIRE ACUTE HOSPITALS NHS TRUST,19457.0,64.98
5,RCB,YORK AND SCARBOROUGH TEACHING HOSPITALS NHS FO...,18486.0,66.33
6,RWD,UNITED LINCOLNSHIRE TEACHING HOSPITALS NHS TRUST,20845.0,66.49
7,RJE,UNIVERSITY HOSPITALS OF NORTH MIDLANDS NHS TRUST,23129.0,67.22
8,RA7,BRISTOL NHS FOUNDATION TRUST,18678.0,68.22
9,RGN,NORTH WEST ANGLIA NHS FOUNDATION TRUST,19362.0,68.55



LARGEST PROVIDER MONTH-ON-MONTH CHANGES


,month,org_code,org_name,total_ae_attendances,previous_month_attendances,attendance_mom_change_pct,four_hour_performance_pct,performance_mom_change_pp
0,2024-10-01,AD913,BECKENHAM BEACON UCC,3624,622,482.64,99.28,0.73
1,2025-11-01,NNJ15,RUTLAND MINOR ILLNESS AND INJURY SERVICE (FORM...,302,53,469.81,100.00,0.00
2,2026-03-01,NNJ15,RUTLAND MINOR ILLNESS AND INJURY SERVICE (FORM...,294,53,454.72,100.00,0.00
3,2024-09-01,AD913,BECKENHAM BEACON UCC,622,3826,-83.74,98.55,-1.24
4,2025-07-01,NNJ0H,LLR EA - THE MERLYN VAZ HEALTH & SOCIAL CARE C...,338,184,83.70,99.70,-0.30
5,2025-12-01,NNJ15,RUTLAND MINOR ILLNESS AND INJURY SERVICE (FORM...,55,302,-81.79,100.00,0.00
6,2025-10-01,RX1,NOTTINGHAM UNIVERSITY HOSPITALS NHS TRUST,25102,14874,68.76,61.17,8.72
7,2026-03-01,L83087,OKEHAMPTON MEDICAL CENTRE,86,52,65.38,100.00,0.00
8,2026-06-01,C82038,LATHAM HOUSE MEDICAL PRACTICE,441,268,64.55,100.00,0.00
9,2024-07-01,NNJ14,OADBY,802,500,60.40,100.00,0.20



SQL provider outputs saved successfully.


In [6]:
# ============================================================
# ADVANCED TYPE 1 A&E SQL BENCHMARKING
# ============================================================

from sqlalchemy import text
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 1. TYPE 1 PROVIDER SUMMARY
# ------------------------------------------------------------

type1_provider_summary_sql = run_sql("""
    WITH provider_totals AS (

        SELECT
            org_code,

            COUNT(DISTINCT month) AS months_reported,

            SUM(type_1_attendances) AS total_type1_attendances,

            SUM(type_1_over_four_hours) AS total_type1_over_four_hours

        FROM ae_type1_provider_month

        WHERE type_1_attendances > 0

        GROUP BY org_code
    )

    SELECT
        p.org_code,

        d.latest_org_name AS org_name,

        p.months_reported,

        p.total_type1_attendances,

        ROUND(
            p.total_type1_attendances::NUMERIC
            / p.months_reported,
            0
        ) AS average_monthly_type1_attendances,

        ROUND(
            (
                (
                    p.total_type1_attendances
                    - p.total_type1_over_four_hours
                )::NUMERIC
                / p.total_type1_attendances
            ) * 100,
            2
        ) AS type1_performance_pct

    FROM provider_totals p

    LEFT JOIN provider_dimension d
        ON p.org_code = d.org_code

    WHERE p.months_reported >= 24

    ORDER BY
        p.total_type1_attendances DESC;
""")


print("=" * 75)
print("TYPE 1 PROVIDER SUMMARY")
print("=" * 75)

display(type1_provider_summary_sql.head(15))


# ============================================================
# 2. SQL CORRELATION: DEMAND VS TYPE 1 PERFORMANCE
# ============================================================

type1_correlation_sql = run_sql("""
    WITH provider_metrics AS (

        SELECT
            org_code,

            COUNT(DISTINCT month) AS months_reported,

            SUM(type_1_attendances) AS total_attendances,

            SUM(type_1_over_four_hours) AS total_over_four_hours

        FROM ae_type1_provider_month

        WHERE type_1_attendances > 0

        GROUP BY org_code
    ),

    calculated_metrics AS (

        SELECT
            org_code,

            total_attendances::NUMERIC
            / months_reported
                AS average_monthly_attendances,

            (
                (
                    total_attendances
                    - total_over_four_hours
                )::NUMERIC
                / total_attendances
            ) * 100
                AS performance_pct

        FROM provider_metrics

        WHERE months_reported >= 24
    )

    SELECT
        COUNT(*) AS providers_included,

        ROUND(
            CORR(
                average_monthly_attendances,
                performance_pct
            )::NUMERIC,
            3
        ) AS demand_performance_correlation

    FROM calculated_metrics;
""")


print("\n" + "=" * 75)
print("TYPE 1 DEMAND VS PERFORMANCE CORRELATION")
print("=" * 75)

display(type1_correlation_sql)


# ============================================================
# 3. BEST HIGH-VOLUME TYPE 1 PROVIDERS
# ============================================================

best_high_volume_type1_sql = run_sql("""
    WITH provider_totals AS (

        SELECT
            org_code,

            COUNT(DISTINCT month) AS months_reported,

            SUM(type_1_attendances) AS total_attendances,

            SUM(type_1_over_four_hours) AS total_over_four_hours

        FROM ae_type1_provider_month

        WHERE type_1_attendances > 0

        GROUP BY org_code
    ),

    provider_metrics AS (

        SELECT
            org_code,

            total_attendances::NUMERIC
            / months_reported
                AS average_monthly_attendances,

            (
                (
                    total_attendances
                    - total_over_four_hours
                )::NUMERIC
                / total_attendances
            ) * 100
                AS performance_pct

        FROM provider_totals

        WHERE months_reported >= 24
    ),

    demand_cutoff AS (

        SELECT
            PERCENTILE_CONT(0.75)
            WITHIN GROUP (
                ORDER BY average_monthly_attendances
            ) AS cutoff

        FROM provider_metrics
    )

    SELECT
        p.org_code,

        d.latest_org_name AS org_name,

        ROUND(
            p.average_monthly_attendances,
            0
        ) AS average_monthly_type1_attendances,

        ROUND(
            p.performance_pct,
            2
        ) AS type1_performance_pct

    FROM provider_metrics p

    CROSS JOIN demand_cutoff c

    LEFT JOIN provider_dimension d
        ON p.org_code = d.org_code

    WHERE
        p.average_monthly_attendances >= c.cutoff

    ORDER BY
        p.performance_pct DESC

    LIMIT 10;
""")


print("\n" + "=" * 75)
print("BEST HIGH-VOLUME TYPE 1 PROVIDERS")
print("=" * 75)

display(best_high_volume_type1_sql)


# ============================================================
# 4. WEAKEST HIGH-VOLUME TYPE 1 PROVIDERS
# ============================================================

weak_high_volume_type1_sql = run_sql("""
    WITH provider_totals AS (

        SELECT
            org_code,

            COUNT(DISTINCT month) AS months_reported,

            SUM(type_1_attendances) AS total_attendances,

            SUM(type_1_over_four_hours) AS total_over_four_hours

        FROM ae_type1_provider_month

        WHERE type_1_attendances > 0

        GROUP BY org_code
    ),

    provider_metrics AS (

        SELECT
            org_code,

            total_attendances::NUMERIC
            / months_reported
                AS average_monthly_attendances,

            (
                (
                    total_attendances
                    - total_over_four_hours
                )::NUMERIC
                / total_attendances
            ) * 100
                AS performance_pct

        FROM provider_totals

        WHERE months_reported >= 24
    ),

    demand_cutoff AS (

        SELECT
            PERCENTILE_CONT(0.75)
            WITHIN GROUP (
                ORDER BY average_monthly_attendances
            ) AS cutoff

        FROM provider_metrics
    )

    SELECT
        p.org_code,

        d.latest_org_name AS org_name,

        ROUND(
            p.average_monthly_attendances,
            0
        ) AS average_monthly_type1_attendances,

        ROUND(
            p.performance_pct,
            2
        ) AS type1_performance_pct

    FROM provider_metrics p

    CROSS JOIN demand_cutoff c

    LEFT JOIN provider_dimension d
        ON p.org_code = d.org_code

    WHERE
        p.average_monthly_attendances >= c.cutoff

    ORDER BY
        p.performance_pct ASC

    LIMIT 10;
""")


print("\n" + "=" * 75)
print("WEAKEST HIGH-VOLUME TYPE 1 PROVIDERS")
print("=" * 75)

display(weak_high_volume_type1_sql)


# ============================================================
# 5. MEANINGFUL TYPE 1 MONTH-ON-MONTH CHANGES
# ============================================================
# A minimum activity threshold removes extreme percentage
# changes caused by very small providers.

meaningful_type1_mom_sql = run_sql("""
    WITH monthly_activity AS (

        SELECT
            month,
            org_code,
            type_1_attendances,
            type_1_performance_pct

        FROM ae_type1_provider_month

        WHERE type_1_attendances > 0
    ),

    monthly_comparison AS (

        SELECT
            month,
            org_code,
            type_1_attendances,

            LAG(type_1_attendances) OVER (
                PARTITION BY org_code
                ORDER BY month
            ) AS previous_month_attendances,

            type_1_performance_pct,

            LAG(type_1_performance_pct) OVER (
                PARTITION BY org_code
                ORDER BY month
            ) AS previous_month_performance

        FROM monthly_activity
    )

    SELECT
        m.month,

        m.org_code,

        d.latest_org_name AS org_name,

        m.type_1_attendances,

        m.previous_month_attendances,

        ROUND(
            (
                (
                    m.type_1_attendances
                    - m.previous_month_attendances
                )::NUMERIC
                / NULLIF(
                    m.previous_month_attendances,
                    0
                )
            ) * 100,
            2
        ) AS attendance_mom_change_pct,

        m.type_1_performance_pct,

        ROUND(
            (
                m.type_1_performance_pct
                - m.previous_month_performance
            )::NUMERIC,
            2
        ) AS performance_mom_change_pp

    FROM monthly_comparison m

    LEFT JOIN provider_dimension d
        ON m.org_code = d.org_code

    WHERE
        m.previous_month_attendances >= 5000
        AND m.type_1_attendances >= 5000

    ORDER BY
        ABS(
            (
                (
                    m.type_1_attendances
                    - m.previous_month_attendances
                )::NUMERIC
                / NULLIF(
                    m.previous_month_attendances,
                    0
                )
            ) * 100
        ) DESC

    LIMIT 20;
""")


print("\n" + "=" * 75)
print("LARGEST MEANINGFUL TYPE 1 MONTH-ON-MONTH CHANGES")
print("=" * 75)

display(meaningful_type1_mom_sql)


# ============================================================
# 6. SAVE SQL ANALYSIS OUTPUTS
# ============================================================

type1_provider_summary_sql.to_csv(
    PROCESSED_PATH / "sql_type1_provider_summary.csv",
    index=False
)

best_high_volume_type1_sql.to_csv(
    PROCESSED_PATH / "sql_best_high_volume_type1.csv",
    index=False
)

weak_high_volume_type1_sql.to_csv(
    PROCESSED_PATH / "sql_weak_high_volume_type1.csv",
    index=False
)

meaningful_type1_mom_sql.to_csv(
    PROCESSED_PATH / "sql_type1_mom_changes.csv",
    index=False
)


# ============================================================
# 7. CREATE REUSABLE SQL SCRIPT IN PROJECT
# ============================================================

SQL_PATH = PROJECT_ROOT / "sql"
SQL_PATH.mkdir(parents=True, exist_ok=True)

sql_script = """
-- ============================================================
-- NHS A&E PROVIDER ANALYSIS
-- PostgreSQL analytical queries
-- ============================================================


-- ------------------------------------------------------------
-- Top 10 busiest providers
-- ------------------------------------------------------------

WITH provider_totals AS (
    SELECT
        org_code,
        SUM(total_ae_attendances) AS total_attendances
    FROM ae_provider_month
    WHERE total_ae_attendances > 0
    GROUP BY org_code
)

SELECT
    org_code,
    total_attendances,
    RANK() OVER (
        ORDER BY total_attendances DESC
    ) AS attendance_rank
FROM provider_totals
ORDER BY attendance_rank
LIMIT 10;


-- ------------------------------------------------------------
-- Type 1 demand vs performance correlation
-- ------------------------------------------------------------

WITH provider_metrics AS (
    SELECT
        org_code,
        COUNT(DISTINCT month) AS months_reported,
        SUM(type_1_attendances) AS total_attendances,
        SUM(type_1_over_four_hours) AS total_over_four_hours
    FROM ae_type1_provider_month
    WHERE type_1_attendances > 0
    GROUP BY org_code
),

calculated_metrics AS (
    SELECT
        org_code,
        total_attendances::NUMERIC / months_reported
            AS average_monthly_attendances,
        (
            (
                total_attendances
                - total_over_four_hours
            )::NUMERIC
            / total_attendances
        ) * 100 AS performance_pct
    FROM provider_metrics
    WHERE months_reported >= 24
)

SELECT
    COUNT(*) AS providers_included,
    CORR(
        average_monthly_attendances,
        performance_pct
    ) AS demand_performance_correlation
FROM calculated_metrics;
"""

(SQL_PATH / "04_provider_analysis.sql").write_text(
    sql_script,
    encoding="utf-8"
)


print("\nSaved SQL outputs successfully.")

print(
    "\nReusable SQL script created:",
    SQL_PATH / "04_provider_analysis.sql"
)

TYPE 1 PROVIDER SUMMARY


,org_code,org_name,months_reported,total_type1_attendances,average_monthly_type1_attendances,type1_performance_pct
0,RRK,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,29,994229.0,34284.0,59.19
1,RAJ,MID AND SOUTH ESSEX NHS FOUNDATION TRUST,29,842163.0,29040.0,65.86
2,RM3,NORTHERN CARE ALLIANCE NHS FOUNDATION TRUST,29,749537.0,25846.0,59.31
3,RYR,UNIVERSITY HOSPITALS SUSSEX NHS FOUNDATION TRUST,29,738684.0,25472.0,59.59
4,R0A,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,29,738043.0,25450.0,57.97
5,R1H,BARTS HEALTH NHS TRUST,29,733663.0,25299.0,55.95
6,RAL,ROYAL FREE LONDON NHS FOUNDATION TRUST,29,680223.0,23456.0,65.84
7,RDU,FRIMLEY HEALTH NHS FOUNDATION TRUST,29,656529.0,22639.0,58.76
8,RWE,UNIVERSITY HOSPITALS OF LEICESTER NHS TRUST,29,640203.0,22076.0,59.16
9,RTG,UNIVERSITY HOSPITALS OF DERBY AND BURTON NHS F...,29,577370.0,19909.0,54.21



TYPE 1 DEMAND VS PERFORMANCE CORRELATION


,providers_included,demand_performance_correlation
0,121,0.038



BEST HIGH-VOLUME TYPE 1 PROVIDERS


...


WEAKEST HIGH-VOLUME TYPE 1 PROVIDERS


,org_code,org_name,average_monthly_type1_attendances,type1_performance_pct
0,RX1,NOTTINGHAM UNIVERSITY HOSPITALS NHS TRUST,13879.0,46.79
1,RJE,UNIVERSITY HOSPITALS OF NORTH MIDLANDS NHS TRUST,14250.0,48.02
2,RJ2,LEWISHAM AND GREENWICH NHS TRUST,15196.0,49.52
3,RWP,WORCESTERSHIRE ACUTE HOSPITALS NHS TRUST,13587.0,50.28
4,R1K,LONDON NORTH WEST UNIVERSITY HEALTHCARE NHS TRUST,13622.0,50.84
5,REM,LIVERPOOL UNIVERSITY HOSPITALS NHS FOUNDATION ...,15843.0,51.22
6,RTG,UNIVERSITY HOSPITALS OF DERBY AND BURTON NHS F...,19909.0,54.21
7,RF4,"BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...",13375.0,55.88
8,R1H,BARTS HEALTH NHS TRUST,25299.0,55.95
9,RJZ,KING'S COLLEGE HOSPITAL NHS FOUNDATION TRUST,15484.0,56.21



LARGEST MEANINGFUL TYPE 1 MONTH-ON-MONTH CHANGES


,month,org_code,org_name,type_1_attendances,previous_month_attendances,attendance_mom_change_pct,type_1_performance_pct,performance_mom_change_pp
0,2026-07-01,RA7,BRISTOL NHS FOUNDATION TRUST,26582,16168,64.41,65.94,-0.22
1,2025-01-01,RAL,ROYAL FREE LONDON NHS FOUNDATION TRUST,24850,17780,39.76,60.37,-7.46
2,2024-05-01,RJL,NORTHERN LINCOLNSHIRE AND GOOLE NHS FOUNDATION...,9065,15035,-39.71,47.83,-25.94
3,2025-01-01,RWD,UNITED LINCOLNSHIRE TEACHING HOSPITALS NHS TRUST,15569,11294,37.85,59.64,20.11
4,2025-04-01,RTK,ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATIO...,6744,10757,-37.31,46.03,-19.62
5,2025-03-01,RHW,ROYAL BERKSHIRE NHS FOUNDATION TRUST,12163,9494,28.11,59.94,-0.92
6,2026-03-01,RTG,UNIVERSITY HOSPITALS OF DERBY AND BURTON NHS F...,21011,16670,26.04,56.32,7.00
7,2026-03-01,RHM,UNIVERSITY HOSPITAL SOUTHAMPTON NHS FOUNDATION...,12105,9612,25.94,70.25,-1.26
8,2024-11-01,RBL,WIRRAL UNIVERSITY TEACHING HOSPITAL NHS FOUNDA...,5970,8011,-25.48,48.06,3.56
9,2024-12-01,RBL,WIRRAL UNIVERSITY TEACHING HOSPITAL NHS FOUNDA...,7469,5970,25.11,45.56,-2.50



Saved SQL outputs successfully.

Reusable SQL script created: /home/jovyan/work/sql/04_provider_analysis.sql
